In [1]:
import tensorflow as tf

print(tf.__version__)

2.21.0


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/karhut_ml_dataset.csv")

df["date"] = pd.to_datetime(df["date"])

df = df.sort_values(["cell_id", "date"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Date:", df["date"].min(), "to", df["date"].max())
print("Cells:", df["cell_id"].nunique())

Shape: (34272, 39)
Date: 2026-05-08 00:00:00 to 2026-09-10 00:00:00
Cells: 272


In [3]:
features = [
    # Fire History
    "fire_count_lag1",
    "fire_count_lag3",
    "fire_count_lag7",
    "frp_sum_lag1",
    "frp_sum_lag3",
    "frp_sum_lag7",
    "fire_count_roll3",
    "fire_count_roll7",
    "frp_sum_roll3",
    "frp_sum_roll7",

    # Weather
    "temperature_mean",
    "temperature_max",
    "relative_humidity_mean",
    "precipitation_sum",
    "boundary_layer_height_mean",
    "vapour_pressure_deficit_mean",
    "wind_speed_mean",
    "wind_speed_max",
    "wind_u_mean",
    "wind_v_mean"
]

target = "target_fire_active_t1"

print("Features:", len(features))

Features: 20


In [4]:
SEQ_LEN = 14

print("Sequence length:", SEQ_LEN)

Sequence length: 14


In [5]:
def create_sequences(data, features, target, seq_len):
    X_seq = []
    y_seq = []
    dates = []
    cells = []

    for cell_id, group in data.groupby("cell_id"):

        group = group.sort_values("date").reset_index(drop=True)

        X_values = group[features].values.astype(np.float32)
        y_values = group[target].values.astype(np.float32)
        date_values = group["date"].values

        for i in range(seq_len, len(group)):

            # Past 14 days
            X_seq.append(
                X_values[i-seq_len:i]
            )

            # Target pada hari berikutnya
            y_seq.append(
                y_values[i]
            )

            dates.append(
                date_values[i]
            )

            cells.append(
                cell_id
            )

    return (
        np.array(X_seq),
        np.array(y_seq),
        np.array(dates),
        np.array(cells)
    )

In [6]:
X_seq, y_seq, seq_dates, seq_cells = create_sequences(
    df,
    features,
    target,
    SEQ_LEN
)

print("X sequence shape:", X_seq.shape)
print("y shape:", y_seq.shape)

X sequence shape: (30464, 14, 20)
y shape: (30464,)


In [7]:
print("First sequence cell:", seq_cells[0])
print("First target date:", seq_dates[0])

print("\nFirst sequence:")
print(X_seq[0].shape)

print("\nTarget:")
print(y_seq[0])

First sequence cell: -0.75_108.0
First target date: 2026-05-22T00:00:00.000000

First sequence:
(14, 20)

Target:
0.0


In [8]:
train_mask = seq_dates < np.datetime64("2026-08-01")

val_mask = (
    (seq_dates >= np.datetime64("2026-08-01")) &
    (seq_dates < np.datetime64("2026-09-01"))
)

test_mask = seq_dates >= np.datetime64("2026-09-01")

X_train_seq = X_seq[train_mask]
y_train_seq = y_seq[train_mask]

X_val_seq = X_seq[val_mask]
y_val_seq = y_seq[val_mask]

X_test_seq = X_seq[test_mask]
y_test_seq = y_seq[test_mask]

print("TRAIN:", X_train_seq.shape, y_train_seq.shape)
print("VAL  :", X_val_seq.shape, y_val_seq.shape)
print("TEST :", X_test_seq.shape, y_test_seq.shape)

TRAIN: (19312, 14, 20) (19312,)
VAL  : (8432, 14, 20) (8432,)
TEST : (2720, 14, 20) (2720,)


In [9]:
print(
    "Train positive rate:",
    y_train_seq.mean()
)

print(
    "Validation positive rate:",
    y_val_seq.mean()
)

print(
    "Test positive rate:",
    y_test_seq.mean()
)

Train positive rate: 0.11899337
Validation positive rate: 0.33918408
Test positive rate: 0.3194853


In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

n_train, timesteps, n_features = X_train_seq.shape

X_train_2d = X_train_seq.reshape(
    -1,
    n_features
)

scaler.fit(X_train_2d)

StandardScaler()

In [11]:
X_train_seq = scaler.transform(
    X_train_seq.reshape(-1, n_features)
).reshape(
    X_train_seq.shape
)

X_val_seq = scaler.transform(
    X_val_seq.reshape(-1, n_features)
).reshape(
    X_val_seq.shape
)

X_test_seq = scaler.transform(
    X_test_seq.reshape(-1, n_features)
).reshape(
    X_test_seq.shape
)

print(X_train_seq.shape)

(19312, 14, 20)


In [12]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_seq.astype(int)
)

class_weights = {
    0: class_weights_array[0],
    1: class_weights_array[1]
}

print(class_weights)

{0: np.float64(0.5675326201951334), 1: np.float64(4.201914708442124)}


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential([
    LSTM(
        64,
        input_shape=(SEQ_LEN, len(features))
    ),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dropout(0.2),

    Dense(1, activation="sigmoid")
])

model.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[]
)

model.summary()

c:\Users\Asus\Documents\skripsi-karhut\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        21,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,873 (93.25 KB)

 Trainable params: 23,873 (93.25 KB)

 Non-trainable params: 0 (0.00 B)

In [14]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

history = model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(
        X_val_seq,
        y_val_seq
    ),
    epochs=50,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 0.4559 - val_loss: 0.3826
Epoch 2/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.4223 - val_loss: 0.4073
Epoch 3/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.4167 - val_loss: 0.3670
Epoch 4/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - loss: 0.4122 - val_loss: 0.4447
Epoch 5/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.4108 - val_loss: 0.3869
Epoch 6/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.4073 - val_loss: 0.3755
Epoch 7/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.3989 - val_loss: 0.3911
Epoch 8/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.3929 - val_loss: 0.3916
Epoch 9/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.3887 - val_loss: 0.3918
Epoch 10/50
302/302 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.3823 - val_loss: 0.3720


In [15]:
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)

val_proba_lstm = model.predict(
    X_val_seq,
    verbose=0
).ravel()

val_pred_lstm = (
    val_proba_lstm >= 0.5
).astype(int)

print("=== LSTM — Validation ===")

print(
    f"PR-AUC   : "
    f"{average_precision_score(y_val_seq, val_proba_lstm):.6f}"
)

print(
    f"Precision: "
    f"{precision_score(y_val_seq, val_pred_lstm):.6f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_val_seq, val_pred_lstm):.6f}"
)

print(
    f"F1       : "
    f"{f1_score(y_val_seq, val_pred_lstm):.6f}"
)

print(
    f"Accuracy : "
    f"{accuracy_score(y_val_seq, val_pred_lstm):.6f}"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_seq, val_pred_lstm))

=== LSTM — Validation ===
PR-AUC   : 0.898397
Precision: 0.664663
Recall   : 0.966783
F1       : 0.787749
Accuracy : 0.823292

Confusion Matrix:
[[4177 1395]
 [  95 2765]]


In [16]:
SEQ_LEN = 7

print("Sequence length:", SEQ_LEN)

Sequence length: 7


In [17]:
X_seq_7, y_seq_7, seq_dates_7, seq_cells_7 = create_sequences(
    df,
    features,
    target,
    SEQ_LEN
)

print("X sequence shape:", X_seq_7.shape)
print("y shape:", y_seq_7.shape)

X sequence shape: (32368, 7, 20)
y shape: (32368,)


In [18]:
train_mask_7 = seq_dates_7 < np.datetime64("2026-08-01")

val_mask_7 = (
    (seq_dates_7 >= np.datetime64("2026-08-01")) &
    (seq_dates_7 < np.datetime64("2026-09-01"))
)

test_mask_7 = seq_dates_7 >= np.datetime64("2026-09-01")

X_train_7 = X_seq_7[train_mask_7]
y_train_7 = y_seq_7[train_mask_7]

X_val_7 = X_seq_7[val_mask_7]
y_val_7 = y_seq_7[val_mask_7]

X_test_7 = X_seq_7[test_mask_7]
y_test_7 = y_seq_7[test_mask_7]

print("TRAIN:", X_train_7.shape)
print("VAL  :", X_val_7.shape)
print("TEST :", X_test_7.shape)

TRAIN: (21216, 7, 20)
VAL  : (8432, 7, 20)
TEST : (2720, 7, 20)


In [19]:
from sklearn.preprocessing import StandardScaler

scaler_7 = StandardScaler()

n_train, timesteps, n_features = X_train_7.shape

scaler_7.fit(
    X_train_7.reshape(-1, n_features)
)

X_train_7 = scaler_7.transform(
    X_train_7.reshape(-1, n_features)
).reshape(X_train_7.shape)

X_val_7 = scaler_7.transform(
    X_val_7.reshape(-1, n_features)
).reshape(X_val_7.shape)

X_test_7 = scaler_7.transform(
    X_test_7.reshape(-1, n_features)
).reshape(X_test_7.shape)

In [20]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

class_weights_array_7 = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_7.astype(int)
)

class_weights_7 = {
    0: class_weights_array_7[0],
    1: class_weights_array_7[1]
}

print(class_weights_7)

{0: np.float64(0.5642853343262939), 1: np.float64(4.388911874224245)}


In [21]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model_lstm_7 = Sequential([
    LSTM(
        64,
        input_shape=(SEQ_LEN, len(features))
    ),

    Dropout(0.3),

    Dense(32, activation="relu"),

    Dropout(0.2),

    Dense(1, activation="sigmoid")
])

model_lstm_7.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=[]
)

model_lstm_7.summary()

c:\Users\Asus\Documents\skripsi-karhut\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_1 (LSTM)                   │ (None, 64)             │        21,760 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 23,873 (93.25 KB)

 Trainable params: 23,873 (93.25 KB)

 Non-trainable params: 0 (0.00 B)

In [22]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping_7 = EarlyStopping(
    monitor="val_loss",
    patience=7,
    restore_best_weights=True
)

history_lstm_7 = model_lstm_7.fit(
    X_train_7,
    y_train_7,
    validation_data=(
        X_val_7,
        y_val_7
    ),
    epochs=50,
    batch_size=64,
    class_weight=class_weights_7,
    callbacks=[early_stopping_7],
    verbose=1
)

Epoch 1/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.4755 - val_loss: 0.3839
Epoch 2/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4361 - val_loss: 0.3955
Epoch 3/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.4298 - val_loss: 0.4147
Epoch 4/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.4243 - val_loss: 0.3973
Epoch 5/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4208 - val_loss: 0.3715
Epoch 6/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.4175 - val_loss: 0.3634
Epoch 7/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4126 - val_loss: 0.3668
Epoch 8/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.4062 - val_loss: 0.3513
Epoch 9/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4063 - val_loss: 0.3697
Epoch 10/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4023 - val_loss: 0.3727
Epoch 11/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 0.4005 - val_loss: 0.3899
Epoch 12/50
332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step

In [23]:
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    confusion_matrix
)


val_proba_lstm_7 = model_lstm_7.predict(
    X_val_7,
    verbose=0
).ravel()

val_pred_lstm_7 = (
    val_proba_lstm_7 >= 0.5
).astype(int)

print("=== LSTM-7 — Validation ===")

print(
    f"PR-AUC   : "
    f"{average_precision_score(y_val_7, val_proba_lstm_7):.6f}"
)

print(
    f"Precision: "
    f"{precision_score(y_val_7, val_pred_lstm_7):.6f}"
)

print(
    f"Recall   : "
    f"{recall_score(y_val_7, val_pred_lstm_7):.6f}"
)

print(
    f"F1       : "
    f"{f1_score(y_val_7, val_pred_lstm_7):.6f}"
)

print(
    f"Accuracy : "
    f"{accuracy_score(y_val_7, val_pred_lstm_7):.6f}"
)

print("\nConfusion Matrix:")
print(confusion_matrix(y_val_7, val_pred_lstm_7))

=== LSTM-7 — Validation ===
PR-AUC   : 0.893025
Precision: 0.696287
Recall   : 0.950699
F1       : 0.803843
Accuracy : 0.842623

Confusion Matrix:
[[4386 1186]
 [ 141 2719]]
